# Material de aula - Redes Neurais e Deep Learning


## <center> Laboratório 3 - Keras: Classificação Binária: determinação de existencia de doença cardíaca </center>

In [2]:
#CÉLULA KE-LIB-01
import numpy as np
import keras as K
import tensorflow as tf
import pandas as pd
import seaborn as sns
import os
from matplotlib import pyplot as plt
%matplotlib inline
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'

G:\Programas\Anaconda\lib\site-packages\h5py\__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


### Definição de RNA para classificação binaria utilizando Keras. A rede terá topologia 18-(10-10)-1 com limite de 500 epocas.
### Dataset de sintomas e existencia de doença cardíaca.

### Formato do dataset: 14 campos por linha, dos quais 13 constituem os preditores e o 14o corresponde ao valor do imóvel (saída)

### [0]  = Numeric - Patient age
### [1]  = Boolean - sex (0=female, 1=male)
### [2]  = Categorical - chest pain [1 4]
### [3]  = Numeric - Blood Pressure
### [4]  = Numeric - Cholesterol
### [5]  = Boolean - blood sugar (0=low, 1=High)
### [6]  = Categorical - electrocardiographic result encoded as (0, 1, 2)
### [7]  = Numeric - maximum heart rate
### [8]  = Boolean - angina (0 = no, 1 = yes). 
### [9]  = Numeric - ST ("S-wave, T-wave") graph depression
### [10] = Categorical - ST metric encoded as (1, 2, 3)
### [11] = Categorical - count of colored fluoroscopy vessels encoded as (0, 1, 2, 3)
### [12] = Categorical - value related to thalassemia encoded as (3, 6, 7)
### [13] = The last value is 0 to 4, where 0 indicates no heart disease and 1 to 4 indicate heart disease of some kind

### # Os valores numericos foram normalizados com operador Min-Max, os booleanos foram convertidos para [-1, 1] e os categóricos para encoding 1-of(N-1). A saída ficou 3m 0 (sem problema) e 1 (com algum problema). Os arquivos do Data set já foram previamente processados e separados

### 1 - Inicialização dos parâmetros. A inicialização fixa permite que os resultados possam ser reproduzidos de forma sistemática para melhorar as análises.

In [3]:
#CÉLULA KE-LIB-02
np.random.seed(4)
tf.set_random_seed(13)

### 2 - Carregamento do dataset e separação dos vetores X e Y

In [4]:
#CÉLULA KE-LIB-03
dfCleveland = pd.read_csv('cleveland_train.csv', header=None)
dfCleveland

X_train = dfCleveland[np.arange(0,18)]
y_train = dfCleveland[18]

dfCleveland = pd.read_csv('cleveland_test.csv', header=None)
dfCleveland

X_test = dfCleveland[np.arange(0,18)]
y_test = dfCleveland[18]

dfCleveland = pd.read_csv('cleveland_validate.csv', header=None)
dfCleveland

X_validate = dfCleveland[np.arange(0,18)]
y_validate = dfCleveland[18]

### 4 - Monte o Modelo

In [5]:
#CÉLULA KE-LIB-04
#Montando a rede neural
tf.logging.set_verbosity(tf.logging.ERROR) #desliga os warnings do tensorflow

#Inicializador
init = K.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=1)

#Criando o otimizador
simple_adadelta = K.optimizers.Adadelta()

#Construindo o modelo (topologia)
X = K.layers.Input(shape=(18,))
net = K.layers.Dense(units=10, kernel_initializer=init, activation='relu')(X)      #1a camada oculta
net = K.layers.Dropout(0.25)(net)                                                  # dropout da camada anterior

net = K.layers.Dense(units=10, kernel_initializer=init, activation='relu')(net)    #2a camada oculta
net = K.layers.Dropout(0.25)(net)                                                  # dropout da camada anterior

net = K.layers.Dense(units=1,  kernel_initializer=init, activation='sigmoid')(net) #Camada de Saída

model = K.models.Model(X, net) #Define o modelo

#Compilando o modelo
model.compile(loss='binary_crossentropy', optimizer=simple_adadelta, metrics=['acc'])

### 5 - Execute o Treinamento

In [6]:
#CÉLULA KE-LIB-05
#Treinamento
batch_size = 8
max_epochs = 2000
print("Iniciando treinamento... ")
h = model.fit(X_train, y_train, batch_size=batch_size, epochs=max_epochs, validation_data=(X_validate, y_validate), verbose=1)
print("Treinamento finalizado \n")

Iniciando treinamento... 
Train on 178 samples, validate on 59 samples
Epoch 1/2000
178/178 [==============================] - 4s 20ms/step - loss: 0.6931 - acc: 0.5393 - val_loss: 0.6931 - val_acc: 0.5085
Epoch 2/2000
178/178 [==============================] - 0s 820us/step - loss: 0.6928 - acc: 0.5449 - val_loss: 0.6931 - val_acc: 0.5085
Epoch 3/2000
178/178 [==============================] - 0s 724us/step - loss: 0.6926 - acc: 0.5449 - val_loss: 0.6930 - val_acc: 0.5085
Epoch 4/2000
178/178 [==============================] - 0s 786us/step - loss: 0.6924 - acc: 0.5449 - val_loss: 0.6930 - val_acc: 0.5085
Epoch 5/2000
178/178 [==============================] - 0s 685us/step - loss: 0.6923 - acc: 0.5449 - val_loss: 0.6929 - val_acc: 0.5085
Epoch 6/2000
178/178 [==============================] - 0s 724us/step - loss: 0.6921 - acc: 0.5449 - val_loss: 0.6928 - val_acc: 0.5085
Epoch 7/2000
178/178 [==============================] - 0s 865us/step - loss: 0.6913 - acc: 0.5449 - val_loss: 0.6

### 6 - Faça a avaliaçao do Modelo

In [6]:
#CÉLULA KE-LIB-06
eval = model.evaluate(X_train, y_train, verbose=0)
print("Erro médio do conjunto de treinamento: Perda {0:.4f}, acuracia {1:.4f}".format(eval[0], eval[1]*100))

eval = model.evaluate(X_test, y_test, verbose=0)
print("Erro médio do conjunto de teste: Perda {0:.4f}, acuracia {1:.4f}".format(eval[0], eval[1]*100))

Erro médio do conjunto de treinamento: Perda 0.0576, acuracia 97.1910
Erro médio do conjunto de teste: Perda 1.1690, acuracia 86.6667


### 7 - Salve o modelo em Arquivo

In [7]:
#CÉLULA KE-LIB-7
# Salvando modelo em arquivo
print("Salvando modelo em arquivo \n")
mp = ".\\cleveland_model.h5"
model.save(mp)

Salvando modelo em arquivo 



### 8 - Faça um teste de operação com o Modelo

In [8]:
# 7. Usando modelo (operação)
np.set_printoptions(precision=4)
unknown = np.array([[0.75, 1, 0, 1, 0, 0.49, 0.27, 1, -1, -1, 0.62, -1, 0.40, 0, 1, 0.23, 1, 0]], dtype=np.float32)
predicted = model.predict(unknown)
print("Usando o modelo para previsão de doença cardíaca para as caracteristicas: ")
print(unknown)
print("\nO valor de previsão diagnóstico é (0=sem doença, 1=com doença): ")
print(predicted)

Usando o modelo para previsão de doença cardíaca para as caracteristicas: 
[[ 0.75  1.    0.    1.    0.    0.49  0.27  1.   -1.   -1.    0.62 -1.
   0.4   0.    1.    0.23  1.    0.  ]]

O valor de previsão diagnóstico é (0=sem doença, 1=com doença): 
[[0.9945]]


### 9 - Faça o pós-processamento para fornecer resposta compreensível

In [9]:
# 8. Pos-processamento
str = ['com doença', 'sem doença']
if (predicted >= 0.5):
    diag = str[0]
else:
    diag = str[1]

print(diag)

com doença
